# Stage 06: Data Preprocessing

Stage 06. **I based it on the lecture notebook.**

In [1]:
# Install missing packages (uncomment and run to install).
# !pip install pandas python-dotenv

In [2]:
from pathlib import Path

# Project root.
ROOT = Path.cwd()
if not (ROOT/".env.example").exists() and (ROOT.parent/".env.example").exists():
    ROOT = ROOT.parent

CHECKS = [
    ("src/cleaning.py", "NEEDED", "fill, drop, and normalize helpers"),
    ("data/raw/prismatic_evoluations_prices.csv", "NEEDED", "Prismatic Evolutions prices"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT/rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    raise FileNotFoundError(f"{missing} needed file(s) missing under {ROOT}")
print("\nAll needed files present.")

Looking in: /Users/ghostof0days/projects/bootcamp/project

  [OK ]  NEEDED    src/cleaning.py                     fill, drop, and normalize helpers
  [OK ]  NEEDED    data/raw/prismatic_evoluations_prices.csv  Prismatic Evolutions prices

All needed files present.


In [3]:
import sys

import pandas as pd
from dotenv import load_dotenv

# Import helpers from src/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import drop_missing, fill_missing_median, normalize_data
from src.config import get_key

# Load `.env`.
load_dotenv(ROOT/".env")

RAW = ROOT/(get_key("DATA_DIR_RAW", "data/raw") or "data/raw")
PROC = ROOT/(get_key("DATA_DIR_PROCESSED", "data/processed") or "data/processed")
PROC.mkdir(parents=True, exist_ok=True)
print("RAW:", RAW.resolve())
print("PROC:", PROC.resolve())

RAW: /Users/ghostof0days/projects/bootcamp/project/data/raw
PROC: /Users/ghostof0days/projects/bootcamp/project/data/processed


## Load raw prices

In [4]:
# Prismatic Evolutions card-by-date prices.
card_prices = pd.read_csv(RAW/"prismatic_evoluations_prices.csv", parse_dates=["date"])
card_prices["market_price"] = pd.to_numeric(card_prices["market_price"], errors="coerce")
print("Original shape:", card_prices.shape)
print("Original NA count:\n", card_prices.isna().sum())
card_prices.head()

Original shape: (61279, 4)
Original NA count:
 card_name       0
rarity          0
market_price    0
date            0
dtype: int64


,card_name,rarity,market_price,date
0,Amarys - 132/131 | Holofoil,Ultra Rare,4.09,2025-01-17
1,Applin (Master Ball Pattern) | Holofoil,Common,9.99,2025-01-17
2,Applin (Poke Ball Pattern) | Holofoil,Common,3.58,2025-01-17
3,Archaludon (Master Ball Pattern) | Holofoil,Rare,20.00,2025-01-17
4,Area Zero Underdepths (Poke Ball Pattern) | Ho...,Uncommon,4.44,2025-01-17


## Apply cleaning functions

In [5]:
# Drop non-positive prices.
positive = card_prices[card_prices["market_price"] > 0].copy()

# Fill gaps where dating is missing with median.
filled = fill_missing_median(positive, ["market_price"])

# Drop columns where majority (50% or more) of values are missing.
dropped = drop_missing(filled, threshold=0.5)

# Scale remaining numeric column.
cleaned = normalize_data(dropped, ["market_price"])

print("Rows dropped for non-positive price:", len(card_prices) - len(positive))
print("Cleaned shape:", cleaned.shape)
print("Cleaned NA count:\n", cleaned.isna().sum())
print("\nOriginal market price statistics:")
print(card_prices["market_price"].describe())
print("\nCleaned market price statistics:")
print(cleaned["market_price"].describe())
cleaned.head()

Rows dropped for non-positive price: 0
Cleaned shape: (61279, 4)
Cleaned NA count:
 card_name       0
rarity          0
market_price    0
date            0
dtype: int64

Original market price statistics:
count    61279.000000
mean        51.546560
std        141.875164
min          2.000000
25%          4.980000
50%         10.100000
75%         34.595000
max       1618.750000
Name: market_price, dtype: float64

Cleaned market price statistics:
count    61279.000000
mean         0.030646
std          0.087753
min          0.000000
25%          0.001843
50%          0.005010
75%          0.020161
max          1.000000
Name: market_price, dtype: float64


,card_name,rarity,market_price,date
0,Amarys - 132/131 | Holofoil,Ultra Rare,0.001293,2025-01-17
1,Applin (Master Ball Pattern) | Holofoil,Common,0.004942,2025-01-17
2,Applin (Poke Ball Pattern) | Holofoil,Common,0.000977,2025-01-17
3,Archaludon (Master Ball Pattern) | Holofoil,Rare,0.011133,2025-01-17
4,Area Zero Underdepths (Poke Ball Pattern) | Ho...,Uncommon,0.001509,2025-01-17


## Save cleaned dataset

In [6]:
cleaned_path = PROC/"prismatic_prices_cleaned.csv"
cleaned.to_csv(cleaned_path, index=False)
print("Saved cleaned CSV:", cleaned_path)

Saved cleaned CSV: /Users/ghostof0days/projects/bootcamp/project/data/processed/prismatic_prices_cleaned.csv


## Documentation

- I filled `market_price` with each column's median.
- I dropped columns whose NA share is above 0.5.
- I dropped rows with non-positive `market_price`.
- I min-max scaleD `market_price` to [0, 1].
- I saved the result to `data/processed/prismatic_prices_cleaned.csv`.
- I added the README Cleaning Strategy section in `project/README.md`.

Assumptions and risks:
- When I median fill, I assumed the missing numbers are not systematically biased due to the MCAR or MAR concepts from lecture.
- I don't need the columns with too many missing values later.
- When I min-max scaled, I assumed that the observed min and max are representative.